<a href="https://colab.research.google.com/github/SirSaltySalmon/hybrid_rnns_reward_learning/blob/main/hybrid_rnns_reward_learning/train_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs, imports, etc.

## Install Hybrid RNNs repo from github


In [1]:
#@title Environment setup (Colab: clone + install | Local: cd to repo root)
import os
from pathlib import Path

def in_colab():
  try:
    import google.colab  # noqa: F401
    return True
  except ImportError:
    return False


def find_repo_root(start=None):
  start = Path(start or Path.cwd()).resolve()
  for p in [start, *start.parents]:
    if (p / 'pyproject.toml').is_file() and (p / 'custom').is_dir():
      return p
  raise FileNotFoundError(f'Repo root not found from {start}')


if in_colab():
  REPO_URL = 'https://github.com/SirSaltySalmon/hybrid_rnns_reward_learning'  # fork with custom/
  # REPO_URL = 'https://github.com/google-deepmind/hybrid_rnns_reward_learning'  # upstream (no custom/)
  # !rm -rf hybrid_rnns_reward_learning
  !git clone {REPO_URL}
  %cd hybrid_rnns_reward_learning
  !pip install .
  %cd ..
  REPO_ROOT = Path('hybrid_rnns_reward_learning').resolve()
else:
  REPO_ROOT = find_repo_root()
  os.chdir(REPO_ROOT)
  print(f'Local — using repo at {REPO_ROOT}')

Local — using repo at C:\Users\salty.THE-AQUARIUM\Desktop\Vinamilk\1. Workbench\Code\hybrid_rnns_reward_learning


## Import hybRNN functions


In [2]:
from hybrid_rnns_reward_learning import bi_rnn
from hybrid_rnns_reward_learning import cogmod
from hybrid_rnns_reward_learning import fit_hyb_rnn
from hybrid_rnns_reward_learning import hyb_rnn_utilities
from hybrid_rnns_reward_learning import rnn
from hybrid_rnns_reward_learning import rnn_config

## Download human dataset from OSF

In [3]:
import urllib
import os
import pandas as pd
import haiku as hk

In [4]:
download_url = 'https://osf.io/download/dw7f6/'
local_path = os.path.join('/tmp', 'openSourceRawDataset.csv')

In [ ]:
urllib.request.urlretrieve(download_url, local_path)

data_df = pd.read_csv(local_path)
print(data_df.head())

## Environment & device (read this first)



Training uses **JAX**. Whether a cell runs on CPU or GPU depends entirely on which Python the notebook kernel is. This notebook supports two environments:

| Environment | Notebook kernel device | How to get GPU for long runs |
|---|---|---|
| **Google Colab** (GPU runtime) | **GPU** — cells train on the GPU directly | Already on GPU; just run the cells |
| **Local Windows** (`hybrnn_venv`) | **CPU only** — good for smoke tests and plotting | Launch the `wsl ...` command shown in each section; progress prints in your terminal |

Native Windows JAX has no CUDA, so on a local machine the kernel always runs on CPU. The GPU lives in a separate WSL2 Linux environment that we drive with small shell scripts. In practice:

- On **Colab**, run every cell normally — `paper_replication` and the memory-window sweep train on the GPU.
- On **local Windows**, use cells for quick checks and plots, and launch the full GPU runs with the `wsl ...` command printed in each section.

Run the next cell to see what the current kernel is using.

In [5]:
#@title Check JAX backend
import jax

print('jax', jax.__version__)
print('backend', jax.default_backend())
print('devices', jax.devices())

if jax.default_backend() == 'gpu':
  print('GPU active — cells will train on the GPU.')
elif in_colab():
  print('Colab on CPU. Switch via Runtime → Change runtime type → GPU, then rerun this cell.')
else:
  print('Local CPU kernel — use small presets here; launch full GPU jobs via the WSL section below.')

jax 0.10.2
backend cpu
devices [CpuDevice(id=0)]
Local CPU kernel — use small presets here; launch full GPU jobs via the WSL section below.


## Local GPU on Windows (WSL2)



**Colab users can skip this section** — your runtime already provides a GPU.

On Windows, GPU training happens inside a WSL2 Ubuntu environment, not in this kernel. You install it once, then launch runs with a `wsl ...` command and watch progress in your terminal.

**Prerequisites:** a recent NVIDIA Windows driver, WSL2 with `Ubuntu-24.04` installed, and `nvidia-smi` working inside WSL.

**One-time setup** creates a Linux venv with `jax[cuda12]`. Run it from PowerShell:

```
wsl -d Ubuntu-24.04 bash custom/setup_wsl_gpu.sh
```

…or run the next cell once with `RUN_SETUP = True`. The two cells below confirm the GPU venv is ready and print the launch command for each experiment. `wsl_run.sh` sets `XLA_PYTHON_CLIENT_PREALLOCATE=false` and `MEM_FRACTION=0.85` so JAX does not grab all VRAM on laptop GPUs.


In [6]:
#@title WSL GPU — one-time setup (local Windows only)
import platform
import subprocess

WSL_DISTRO = 'Ubuntu-24.04'
RUN_SETUP = True  # set True ONCE to install jax[cuda12] in WSL (~a few minutes), then set back to False

if in_colab():
  print('Colab — GPU is provided by the runtime; WSL setup does not apply.')
elif platform.system() != 'Windows':
  print('Not Windows — on Linux, pip install "jax[cuda12]" in a venv with an NVIDIA GPU.')
elif RUN_SETUP:
  print('Running WSL GPU setup (may take several minutes)...')
  r = subprocess.run(
      ['wsl', '-d', WSL_DISTRO, 'bash', 'custom/setup_wsl_gpu.sh'],
      capture_output=True, text=True, timeout=1800)
  if r.stdout:
    print(r.stdout, end='')
  if r.stderr:
    print(r.stderr, end='')
  r.check_returncode()
  print('Setup finished.')
else:
  print('RUN_SETUP is False (default). To install the GPU venv, set RUN_SETUP = True and rerun,')
  print(f'or run in PowerShell:  wsl -d {WSL_DISTRO} bash custom/setup_wsl_gpu.sh')


Running WSL GPU setup (may take several minutes)...
==> Repo: /mnt/c/Users/salty.THE-AQUARIUM/Desktop/Vinamilk/1. Workbench/Code/hybrid_rnns_reward_learning
==> Venv: /home/salty/hybrnn_venv_wsl (Linux filesystem â€” required for CUDA wheels)
==> GPU:
NVIDIA GeForce RTX 3070 Ti Laptop GPU, 610.47, 8192 MiB
==> Installing uv (user-local, no sudo)...
installing to /home/salty/.local/bin
  uv
  uvx
everything's installed!
==> Installing project + JAX CUDA 12...
==> Verifying JAX GPU...
jax 0.10.2
backend gpu
devices [CudaDevice(id=0)]
GPU matmul OK

Setup complete.
Venv path saved to custom/.wsl_venv_path
Run training:
  wsl -d Ubuntu-24.04 bash custom/wsl_run.sh debug
downloading uv 0.11.25 x86_64-unknown-linux-gnu
Using Python 3.12.3 environment at: /home/salty/hybrnn_venv_wsl
Resolved 30 packages in 343ms
   Building hybrid-rnns-reward-learning @ file:///mnt/c/Users/salty.THE-AQUARIUM/Desktop/Vinamilk/1.%20Workbench/Code/hybrid_rnns_reward_learning
      Built hybrid-rnns-reward-learni

In [7]:
#@title WSL GPU — probe venv & show launch commands (local Windows only)
import platform
import subprocess

WSL_DISTRO = 'Ubuntu-24.04'
RUN_CMDS = [
    f'wsl -d {WSL_DISTRO} bash custom/wsl_run.sh debug',
    f'wsl -d {WSL_DISTRO} bash custom/wsl_run.sh paper_replication',
    f'wsl -d {WSL_DISTRO} bash custom/memory_window/run_sweep.sh smoke',
]

if in_colab():
  print('Colab — use the GPU runtime; WSL does not apply.')
elif platform.system() != 'Windows':
  print('Not Windows — use a Linux venv with pip install "jax[cuda12]" if you have an NVIDIA GPU.')
else:
  print('Launch GPU runs from PowerShell (output appears in that terminal):')
  for cmd in RUN_CMDS:
    print(f'  {cmd}')
  # Call a script file: wsl.exe strips $VAR in inline `bash -c` strings on Windows.
  probe = subprocess.run(
      ['wsl', '-d', WSL_DISTRO, 'bash', 'custom/wsl_probe.sh'],
      capture_output=True, text=True, timeout=120)
  if probe.returncode == 0:
    print('\nWSL JAX probe (expect backend gpu):')
    print(probe.stdout.strip())
  else:
    print('\nWSL GPU venv not ready — run the one-time setup above first.')
    if probe.stdout.strip():
      print(probe.stdout.strip())
    if probe.stderr.strip():
      print(probe.stderr.strip())

Launch GPU runs from PowerShell (output appears in that terminal):
  wsl -d Ubuntu-24.04 bash custom/wsl_run.sh debug
  wsl -d Ubuntu-24.04 bash custom/wsl_run.sh paper_replication
  wsl -d Ubuntu-24.04 bash custom/memory_window/run_sweep.sh smoke

WSL JAX probe (expect backend gpu):
0.10.2
gpu
[CudaDevice(id=0)]


# Set up training config

## Example 1: Fit "Best RL" model (free parameters: α, β, b, κ, Qinit, p)


In [ ]:
config = rnn_config.get_config()
config.model_name = 'cogmod'
config.rnn_rl_params.fit_alpha = True
config.rnn_rl_params.fit_beta = True
config.rnn_rl_params.fit_bias = True
config.rnn_rl_params.fit_forget = True
config.rnn_rl_params.fit_init_v = True
config.rnn_rl_params.fit_persev_t = True
config.rnn_rl_params.fit_init_h = False
config.rnn_rl_params.fit_persev_p = False
config.rnn_rl_params.fit_w = False

## Example 2: Fit winning "hybRNN" model


In [8]:
config = rnn_config.get_config()
config.model_name = 'birnn'
config.rnn_rl_params.w_v = 1
config.rnn_rl_params.w_h = 1
config.rnn_rl_params.fit_forget = True
config.rnn_rl_params.o = False
config.rnn_rl_params.s = True
config.rnn_rl_params.zero_values = True
config.rnn_rl_params.fit_init_v = True
config.rnn_rl_params.fit_init_h = True

In [9]:
config.dataset_path = local_path
config.n_training_steps = 1001
config.batch_size = 32
# config

# Original paper training

## Run the training loop

In [ ]:
scalars, params = fit_hyb_rnn.train(config)

## Inspect results


## Inspect the loss


In [ ]:
scalars

## Inspect fitted model parameters


In [ ]:
params

# Custom experiments (`custom/` package)



The fork-only [`custom/`](https://github.com/SirSaltySalmon/hybrid_rnns_reward_learning/tree/main/custom) package defines named training setups under `custom/experiments/`. Each experiment applies architecture + hyperparameter overrides on top of upstream defaults, then trains with **paper-style accuracy logging** (`paper_acc` % on held-out minibatches).

| Experiment | Model | Steps | Paper target |
|---|---|---|---|
| `debug` | BiRNN | 100 | — (smoke test) |
| `birnn_memory` | BiRNN | 1,001 | — (notebook demo scale) |
| `paper_replication` | Memory-ANN (BiRNN) | 1,000,000 | ~68.3% |

**`paper_replication`** matches Eckstein et al. (Nat Hum Behav, 2026) Methods: lr=1e-4, weight decay=1e-5, hidden=32, batch=128, 1M steps. Expect several hours on GPU (much longer on CPU). Use `COLAB_SMOKE_TEST` below for a quick sanity check before the full run.

**Where this runs:** on Colab the cell below trains on the GPU. On local Windows it runs on the **CPU** kernel — fine for `debug`, far too slow for the full `paper_replication`. For the full GPU run on Windows, launch it via WSL instead (output prints in your terminal):

```
wsl -d Ubuntu-24.04 bash custom/wsl_run.sh paper_replication
```

## Custom experiments — choose experiment


In [ ]:
import sys
from pathlib import Path

repo_root = REPO_ROOT if 'REPO_ROOT' in dir() else Path('hybrid_rnns_reward_learning').resolve()
if str(repo_root) not in sys.path:
  sys.path.insert(0, str(repo_root))

from custom.config import build_config, list_experiments
from custom.io import save_run
from custom.train import default_log_every, train

EXPERIMENT = 'paper_replication'  # debug | birnn_memory | paper_replication
SAVE_RESULTS = True
COLAB_SMOKE_TEST = False  # True → 500 steps for paper_replication only

print('Available experiments:')
for name, description in list_experiments():
  suffix = f' — {description}' if description else ''
  print(f'  {name}{suffix}')

## Run custom experiment


In [ ]:
config, experiment = build_config(EXPERIMENT)
config.dataset_path = local_path  # from dataset download cell above

if COLAB_SMOKE_TEST and EXPERIMENT == 'paper_replication':
  config.n_training_steps = 500
  print('COLAB_SMOKE_TEST: shortened to 500 steps (full paper run = 1,000,000)')

log_every = getattr(experiment, 'LOG_EVERY', default_log_every(config))
target_acc = getattr(experiment, 'TARGET_ACC_PCT', None)

print(f'Running: {EXPERIMENT}')
print(f'  model: {config.model_name}')
print(f'  steps: {config.n_training_steps}  batch: {config.batch_size}')
print(f'  device: {jax.default_backend()}')
if target_acc is not None:
  print(f'  paper target (held-out): ~{target_acc}% trial-wise accuracy')

if (jax.default_backend() != 'gpu' and EXPERIMENT == 'paper_replication'
    and config.n_training_steps > 5000):
  print('  NOTE: CPU kernel + full paper run will take a very long time.')
  print('        Use COLAB_SMOKE_TEST, or run on GPU via WSL:')
  print('        wsl -d Ubuntu-24.04 bash custom/wsl_run.sh paper_replication')

scalars, params = train(config, log_every=log_every)

if SAVE_RESULTS:
  save_run(EXPERIMENT, config, scalars, params=params)

## Inspect custom experiment results


In [ ]:
scalars

# Experiment — How far back does memory reach?


Train Memory-ANN with a **sliding memory window** `N`: predicting the action at trial `t` uses only trials `[t-N, t-1]` (anything older is ignored). Sweep `N ∈ {3,5,8,12,20,35,50,75,100,150}` × 5 seeds, measure held-out accuracy, and find the elbow.

Code lives in the fork-only [`custom/memory_window/`](https://github.com/SirSaltySalmon/hybrid_rnns_reward_learning/tree/main/custom/memory_window) package.

| Preset | N grid | seeds | steps/run |
|---|---|---|---|
| `smoke` | {5, 50} | 1 | 500 |
| `one` | {20} | 1 | 2,000 |
| `all` | full grid | 1 | 10,000 |
| `all_smoke` | full grid | 1 | 500 |
| `full` | full grid | 5 | 1,000,000 |

**Dataset:** cells below load the OSF CSV via `local_path` and pass pre-split tensors into the sweep. WSL / CLI (`run_sweep.sh`, `sweep.py`) load `data/openSourceHumDataset.csv` from `base_config()` unless you override the path — place the same human dataset there for comparable results.

**Where this runs:** on Colab the sweep cell below trains on the GPU. On local Windows it runs on the **CPU** kernel — keep the preset small (`smoke` / `one`). For the full multi-seed GPU sweep on Windows, launch it via WSL (output prints in your terminal):

```
wsl -d Ubuntu-24.04 bash custom/memory_window/run_sweep.sh full
```

Plot afterwards from the saved CSV: `python custom/memory_window/plot.py <results.csv>`


## Memory-window sweep — load data (uses OSF download above)


In [ ]:
from custom.memory_window import sweep as mw_sweep
from custom.memory_window import train_windowed

# Point the loader at the OSF dataset downloaded earlier (local_path).
_data_cfg = mw_sweep.make_config(n=5, seed=0, n_training_steps=1)
_data_cfg.dataset_path = local_path
mw_data = train_windowed.load_data(_data_cfg)
print('Loaded blocks — train: {}  valid: {}  test: {}'.format(
    mw_data['train_dat'].shape[0],
    mw_data['valid_dat'].shape[0],
    mw_data['test_dat'].shape[0]))

## Run the sweep


In [ ]:
PRESET = 'smoke'  # 'smoke' | 'one' | 'all' | 'all_smoke' | 'full'  (full = 50 runs x 1e6 steps)

# On a local CPU kernel keep PRESET small; run 'full' on GPU via:
#   wsl -d Ubuntu-24.04 bash custom/memory_window/run_sweep.sh full
results_path, rows = mw_sweep.run(preset=PRESET, data=mw_data)
results_path

## Plot accuracy vs N and find the elbow


In [ ]:
from custom.memory_window import plot as mw_plot

out_png, elbow = mw_plot.plot(results_path)
print(f'Estimated memory-reach elbow: N={elbow}')

from IPython.display import Image
Image(filename=out_png)